# LangGraph Flow Demo

Run a single ticket through the graph and inspect the trace.

In [ ]:
import os, sys
os.environ.setdefault('LLM_PROVIDER','mock')
sys.path.insert(0,'..')

In [ ]:
from src.utils.schemas import Ticket
from src.utils.helpers import read_json, project_path, app_config
tix = [Ticket(**t) for t in read_json(project_path(app_config()['paths']['tickets']))]
print([t.ticket_id for t in tix])

In [ ]:
from src.graph.support_graph import build_support_graph
from src.graph.graph_state import init_state
from src.retrieval.retriever import Retriever
from src.memory.conversation_memory import ConversationMemory
from src.memory.customer_thread_store import CustomerThreadStore
from src.logging.trace_logger import TraceLogger
from src.utils.llm_client import get_llm_client

client=get_llm_client(); retr=Retriever(); retr.build_index()
mem=ConversationMemory(CustomerThreadStore())
graph=build_support_graph(); print('backend:', getattr(graph,'backend','fallback'))

In [ ]:
t = [x for x in tix if x.ticket_id=='TCK-1002'][0]
tr=TraceLogger(t.ticket_id)
state=init_state(t, {'client':client,'retriever':retr,'memory':mem,'tracer':tr})
final=graph.invoke(state)
print('ROUTE:', final['route'], '| reason:', final['route_reason'])
print(tr.pretty())

In [ ]:
print(final['draft'].draft_reply)